In [2]:
import os
import requests
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

# 1. 載入 .env 檔案中的隱藏帳密
load_dotenv()
USERNAME = os.getenv('SPACETRACK_USER')
PASSWORD = os.getenv('SPACETRACK_PASS')

def run_step1_checkpoint():
    print(f"==================================================")
    print(f"🚀 LEO 風險監測平台 - 本機 API 連線檢查站")
    print(f"   當前檢測時間 (UTC): {datetime.utcnow()}")
    print(f"==================================================\n")

    # ────────────────────────────────────────────────
    # 測試任務 A：NOAA SWPC 空間天氣 API (免驗證接口)
    # ────────────────────────────────────────────────
    print("🌤️ [測試 A] 正在連線 NOAA 空間天氣即時狀態庫...")
    noaa_url = "https://services.swpc.noaa.gov/products/noaa-scales.json"
    
    try:
        noaa_res = requests.get(noaa_url, timeout=10)
        if noaa_res.status_code == 200:
            noaa_data = noaa_res.json()
            print(f"  └─ [成功] 順利對接 NOAA！最新一筆觀測狀態為：")
            print(f"     • {noaa_data[0].get('metric')}: Scale {noaa_data[0].get('scale')}")
        else:
            print(f"  └─ [失敗] NOAA 回傳狀態碼: {noaa_res.status_code}")
    except Exception as e:
        print(f"  └─ [異常] NOAA 連線超時或錯誤: {e}")

    print("\n" + "-"*50 + "\n")

    # ────────────────────────────────────────────────
    # 測試任務 B：Space-Track 官網 Session 登入與資料提取
    # ────────────────────────────────────────────────
    print("🛰️ [測試 B] 正在向美國太空軍 Space-Track 發送身分驗證...")
    
    if not USERNAME or not PASSWORD or USERNAME == "你的註冊信箱@gmail.com":
        print("  └─ ❌ [中結] 尚未填寫真實的 .env 帳號密碼，請填寫後重試！")
        return

    # 建立 Session 保持 Cookie 登入狀態
    site_session = requests.Session()
    login_url = "https://www.space-track.org/ajaxauth/login"
    
    try:
        # 發送登入請求
        login_res = site_session.post(login_url, data={'identity': USERNAME, 'password': PASSWORD}, timeout=15)
        
        if login_res.status_code == 200:
            print("  └─ [登入成功] Session Cookie 取得！正在試撈 5 筆低軌太空垃圾 TLE...")
            
            # 透過同一個 session，試撈 Cosmos 2251 碎片的歷史軌道快照
            test_query_url = "https://www.space-track.org/basicspacedata/query/class/gp/OBJECT_TYPE/DEBRIS/NORAD_CAT_ID/33000--33010/format/json/limit/5"
            debris_res = site_session.get(test_query_url, timeout=15)
            
            if debris_res.status_code == 200:
                deb_data = debris_res.json()
                print(f"  └─ [提取成功] 順利抓到 {len(deb_data)} 筆低軌垃圾物件！")
                print(f"     • 樣本範例名稱: {deb_data[0].get('OBJECT_NAME')}")
                print(f"     • 樣本近地點(PERIGEE): {deb_data[0].get('PERIAPSIS')} km")
            else:
                print(f"  └─ [提取失敗] 資料查詢語法錯誤，狀態碼: {debris_res.status_code}")
        else:
            print(f"  └─ [登入失敗] Space-Track 帳號密碼驗證未過，狀態碼: {login_res.status_code}")
            
    except Exception as e:
        print(f"  └─ [異常] Space-Track 驗證或連線發生錯誤: {e}")

    print(f"\n==================================================")
    print(f"🏁 檢測站通關完畢。若上方兩項皆顯示 [成功]，你的本機網路已完全打通！")
    print(f"==================================================")

if __name__ == "__main__":
    run_step1_checkpoint()

🚀 LEO 風險監測平台 - 本機 API 連線檢查站
   當前檢測時間 (UTC): 2026-06-20 11:00:23.797321

🌤️ [測試 A] 正在連線 NOAA 空間天氣即時狀態庫...


C:\Users\User\AppData\Local\Temp\ipykernel_6928\2656293587.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(f"   當前檢測時間 (UTC): {datetime.utcnow()}")


  └─ [成功] 順利對接 NOAA！最新一筆觀測狀態為：
  └─ [異常] NOAA 連線超時或錯誤: 0

--------------------------------------------------

🛰️ [測試 B] 正在向美國太空軍 Space-Track 發送身分驗證...
  └─ [登入成功] Session Cookie 取得！正在試撈 5 筆低軌太空垃圾 TLE...
  └─ [提取成功] 順利抓到 5 筆低軌垃圾物件！
     • 樣本範例名稱: IUE DEB
     • 樣本近地點(PERIGEE): 22745.773 km

🏁 檢測站通關完畢。若上方兩項皆顯示 [成功]，你的本機網路已完全打通！


## GFZ太空天氣KP&F10.5資料撈取

In [40]:
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone
print("==================================================")
print("🌤️ LEO 平台核心 - 德國 GFZ Potsdam 雙指標 ETL 站")
print(f"當前檢測時間 (UTC): {datetime.now(timezone.utc)}")
print("==================================================\n")

# 1. 自動計算過去 30 天範圍 (GFZ 規定的 ISO-8601 格式)
now_utc = datetime.now(timezone.utc)
start_dt = (now_utc - timedelta(days=30)).strftime('%Y-%m-%dT00:00:00Z')
end_dt = now_utc.strftime('%Y-%m-%dT23:59:59Z')

kp_url = f"https://kp.gfz-potsdam.de/app/json/?start={start_dt}&end={end_dt}&index=Kp"
f107_url = f"https://kp.gfz-potsdam.de/app/json/?start={start_dt}&end={end_dt}&index=Fobs"

try:
    # 2. Extract (同步抽取 GFZ 兩大 API)
    print("📡 正在向德國 GFZ 伺服器請求 30 天 Kp 指數 JSON...")
    kp_res = requests.get(kp_url, timeout=15)
    print("📡 正在向德國 GFZ 伺服器請求 30 天 F10.7 指數 JSON...")
    f107_res = requests.get(f107_url, timeout=15)

    if kp_res.status_code == 200 and f107_res.status_code == 200:
        print("✅ 【大成功】！GFZ 雙水源 JSON 成功獲取。")
        
        kp_data = kp_res.json()
        f107_data = f107_res.json()

        # 3. Transform - 展平並建立獨立 DataFrame
        df_kp = pd.DataFrame({
            'timestamp_utc': pd.to_datetime(kp_data['datetime']),
            'kp_index': kp_data['Kp']
        })
        
        df_f107 = pd.DataFrame({
            'timestamp_utc': pd.to_datetime(f107_data['datetime']),
            'f107_value': f107_data['Fobs']
        })

        # 提取日期
        df_kp['date'] = df_kp['timestamp_utc'].dt.strftime('%Y-%m-%d')
        df_f107['date'] = df_f107['timestamp_utc'].dt.strftime('%Y-%m-%d')

        # 4. 執行你指定的業務聚合算分 (Aggregation & Normalization)
        # Kp 取每日最大值以捕捉暴風尖峰
        df_kp_daily = df_kp.groupby('date')['kp_index'].max().reset_index()
        df_kp_daily.columns = ['date', 'datekp_max']

        # F10.7 取每日平均值以捕捉背景輻射基礎波動
        df_f107_daily = df_f107.groupby('date')['f107_value'].mean().reset_index()
        df_f107_daily.columns = ['date', 'datef107_mean']

        # 將兩份 GFZ 數據一對一 Join
        df_merged = pd.merge(df_kp_daily, df_f107_daily, on='date')

        # 特徵工程：將 Kp 連續資料正規化為 0~100 的 dategst_score
        def map_kp_to_g_scale(kp):
                if kp >= 9.0: return 5
                elif kp >= 8.0: return 4
                elif kp >= 7.0: return 3
                elif kp >= 6.0: return 2
                elif kp >= 5.0: return 1
                else: return 0

        df_merged['dategst_score'] = df_merged['datekp_max'].apply(map_kp_to_g_scale)

        
        
        # 生成資料庫黃金主鍵 date_key (YYYYMMDD)
        df_merged['date_key'] = pd.to_datetime(df_merged['date']).dt.strftime('%Y%m%d').astype(int)

        # 整理欄位排序
        df_final = df_merged[['date_key', 'date', 'datekp_max', 'datef107_mean', 'dategst_score']].sort_values(by='date_key')

        print("\n📊 【GFZ Potsdam 雙指標融合時序事實表】")
        print("-" * 75)
        print(df_final.to_string(index=False))
        print("-" * 75)

        # 5. 落存清洗後Fact資料
        df_final.to_csv("fact_gfz_weather_clean.csv", index=False)
        print("💾 歷史黃金事實表已落存本地：【fact_gfz_weather_clean.csv】")
        print("==================================================")
        print("🏁 這就是你最終決定最完美的頂級架構！數據已完全就緒。")
        print("==================================================")

    else:
        print(f"❌ 請求失敗，狀態碼: Kp={kp_res.status_code}, F107={f107_res.status_code}")
except Exception as e:
    print(f"❌ 執行管線發生連線或解析異常: {e}")

🌤️ LEO 平台核心 - 德國 GFZ Potsdam 雙指標 ETL 站
當前檢測時間 (UTC): 2026-06-21 13:46:39.018487+00:00

📡 正在向德國 GFZ 伺服器請求 30 天 Kp 指數 JSON...
📡 正在向德國 GFZ 伺服器請求 30 天 F10.7 指數 JSON...
✅ 【大成功】！GFZ 雙水源 JSON 成功獲取。

📊 【GFZ Potsdam 雙指標融合時序事實表】
---------------------------------------------------------------------------
 date_key       date  datekp_max  datef107_mean  dategst_score
 20260522 2026-05-22       2.333          124.4              0
 20260523 2026-05-23       1.000          137.1              0
 20260524 2026-05-24       2.333          132.8              0
 20260525 2026-05-25       2.667          139.0              0
 20260526 2026-05-26       3.667          140.8              0
 20260527 2026-05-27       3.000          141.7              0
 20260528 2026-05-28       3.667          144.9              0
 20260529 2026-05-29       4.000          148.1              0
 20260530 2026-05-30       4.000          141.6              0
 20260531 2026-05-31       3.333          135.6              0
 20260601 20

In [34]:
import os
import requests
import json
import pandas as pd
from dotenv import load_dotenv

# 1. 載入 .env 檔案中的安全憑證
load_dotenv()

IDENTITY_EMAIL = os.getenv("SPACETRACK_USER")
IDENTITY_PASSWORD = os.getenv("SPACETRACK_PASS")

if not IDENTITY_EMAIL or not IDENTITY_PASSWORD:
    raise ValueError("❌ 錯誤：請確認你的 .env 中有定義 SPACE_TRACK_EMAIL 與 SPACE_TRACK_PASSWORD")

# 2. 初始化維持 Cookie 狀態的安全 Session
session = requests.Session()
login_url = "https://www.space-track.org/ajaxauth/login"
login_payload = {
    "identity": IDENTITY_EMAIL,
    "password": IDENTITY_PASSWORD
}

print("📡 正在向美國太空軍 Space-Track 進行身分驗證...")
login_res = session.post(login_url, data=login_payload, timeout=15)

if login_res.status_code != 200 or "LOGIN_FAILED" in login_res.text:
    print(f"❌ 登入失敗！請檢查帳密。")
else:
    print("✅ 驗證成功！Session Cookie 已鎖定。")
    
    # 3. 使用官方文件第 10 頁規範的 LEO 暢通 API 路徑
    query_url = "https://www.space-track.org/basicspacedata/query/class/gp/MEAN_MOTION/>11.25/decay_date/null-val/limit/3"
    
    print("📡 正在直通美軍伺服器撈取真實 LEO TLE JSON 契約...")
    data_res = session.get(query_url, timeout=20)
    
    if data_res.status_code == 200:
        real_raw_json = data_res.json()
        print(f"🎉 【完美通關】！成功抓到 {len(real_raw_json)} 筆真實原始 JSON！\n")
        
        # 4. 轉換為 DataFrame 現場秀出真正的欄位
        df_real = pd.DataFrame(real_raw_json)
        
        print("📝 【美軍 Space-Track 官方原生欄位檢查】")
        print("-" * 75)
        # 顯示最核心的星體名稱、編號以及用來算高度的幾何欄位
        display(df_real[['OBJECT_NAME', 'NORAD_CAT_ID', 'OBJECT_TYPE', 'APOAPSIS', 'PERIAPSIS']])
        print("-" * 75)
        
        # 5. 特徵工程現場實測：轉浮點數並精算出幾何平均高度
        df_real['APOAPSIS'] = pd.to_numeric(df_real['APOAPSIS'], errors='coerce')
        df_real['PERIAPSIS'] = pd.to_numeric(df_real['PERIAPSIS'], errors='coerce')
        df_real['current_altitude_km'] = (df_real['APOAPSIS'] + df_real['PERIAPSIS']) / 2.0
        
        print("\n🚀 【Jupyter Notebook 現場真實高度精算結果】")
        display(df_real[['OBJECT_NAME', 'OBJECT_TYPE', 'current_altitude_km']])
        
    else:
        print(f"❌ 抓取失敗！HTTP 狀態碼: {data_res.status_code}")
        print(f"伺服器錯誤回報: {data_res.text}")

📡 正在向美國太空軍 Space-Track 進行身分驗證...
✅ 驗證成功！Session Cookie 已鎖定。
📡 正在直通美軍伺服器撈取真實 LEO TLE JSON 契約...
🎉 【完美通關】！成功抓到 3 筆真實原始 JSON！

📝 【美軍 Space-Track 官方原生欄位檢查】
---------------------------------------------------------------------------


KeyError: "['APOAPSIS', 'PERIAPSIS'] not in index"

In [43]:
import os
import requests
import pandas as pd
import time
from datetime import datetime, timezone, timedelta
from dotenv import load_dotenv

# 1. 安全驗證
load_dotenv()
IDENTITY_EMAIL = os.getenv("SPACETRACK_USER")
IDENTITY_PASSWORD = os.getenv("SPACETRACK_PASS")

session = requests.Session()
login_url = "https://www.space-track.org/ajaxauth/login"
login_payload = {"identity": IDENTITY_EMAIL, "password": IDENTITY_PASSWORD}

print("📡 正在向 Space-Track 進行安全身分驗證...")
login_res = session.post(login_url, data=login_payload, timeout=15)

if login_res.status_code != 200 or "LOGIN_FAILED" in login_res.text:
    print("❌ 登入失敗！")
else:
    print("✅ 驗證成功！啟動 7 天每日定點抽樣管線...\n")
    
    fact_rows = []
    now_utc = datetime.now(timezone.utc)
    
    # 2. 核心時序探針：一天一天查，每天只查中午 12:00 ~ 13:00 的一小時快照
    for d in range(7):
        target_day = now_utc - timedelta(days=d)
        date_str = target_day.strftime('%Y-%m-%d')
        dk = int(target_day.strftime('%Y%m%d'))
        
        # 構造當天中午的嚴格時間窗口
        start_window = f"{date_str}%2000:00:00"
        end_window = f"{date_str}%2023:59:59"
        
        # API 網址：限制在這一小時內、LEO 物體、未墜毀
        query_url = f"https://www.space-track.org/basicspacedata/query/class/gp_history/EPOCH/{start_window}--{end_window}/MEAN_MOTION/>11.25/decay_date/null-val/orderby/NORAD_CAT_ID%20asc"
        
        print(f"📅 正在狙擊日期 [{date_str}] 中午的 LEO 軌道快照...")
        data_res = session.get(query_url, timeout=30)
        
        if data_res.status_code == 200:
            day_json = data_res.json()
            df_day = pd.DataFrame(day_json)
            
            if df_day.empty:
                print(f"⚠️ {date_str} 該時段無雷達觀測資料，跳過。")
                continue
                
            print(f"  -> 成功抓取 {len(df_day)} 筆在軌物體狀態，開始分桶計算...")
            
            # 數據特徵工程與分桶
            df_day['APOAPSIS'] = pd.to_numeric(df_day['APOAPSIS'], errors='coerce')
            df_day['PERIAPSIS'] = pd.to_numeric(df_day['PERIAPSIS'], errors='coerce')
            df_day['alt'] = (df_day['APOAPSIS'] + df_day['PERIAPSIS']) / 2.0
            
            # 因為是一顆一顆星查，這裡直接使用 drop_duplicates 確保每顆星在當天中午只留最新的一筆！
            df_day = df_day.sort_values('EPOCH').drop_duplicates(subset=['NORAD_CAT_ID'], keep='last')
            
            shell_bins = [200, 400, 600, 800, 2000]
            shell_labels = [1, 2, 3, 4]
            df_day['shell_id'] = pd.cut(df_day['alt'], bins=shell_bins, labels=shell_labels)
            
            # 按殼層統計總量
            for shell in shell_labels:
                group = df_day[df_day['shell_id'] == shell]
                sat_count = int(group[group['OBJECT_TYPE'] == 'PAYLOAD'].shape[0])
                debris_count = int(group[group['OBJECT_TYPE'] != 'PAYLOAD'].shape[0])
                total_count = sat_count + debris_count
                density_score = int(min(100, (total_count / 2000.0) * 100)) # 歸一化分
                
                fact_rows.append({
                    "date_key": dk,
                    "shell_id": shell,
                    "satellite_count": sat_count,
                    "debris_count": debris_count,
                    "total_count": total_count,
                    "density_score": density_score
                })
        else:
            print(f"❌ 抓取 {date_str} 失敗，狀態碼: {data_res.status_code}")
            
        # 遵守官方守冊第 2 頁規範：每次請求完稍微冷卻 1 秒，做個優雅的資料公民
        time.sleep(1)
        
    # 3. 組合最終完美的 7 天大盤歷史事實表
    df_final_fact = pd.DataFrame(fact_rows).sort_values(by=['date_key', 'shell_id'], ascending=[False, True])
    
    print("\n📊 【大功告成！透過 7天定點動態抽樣 洗出的 LEO 殼層擁擠事實表】")
    print("-" * 90)
    display(df_final_fact)
    print("-" * 90)
    
    df_final_fact.to_csv("fact_orbital_density_sampled_7days.csv", index=False)
    print("💾 完美的 7 天連續歷史事實表已落檔：【fact_orbital_density_sampled_7days.csv】")

📡 正在向 Space-Track 進行安全身分驗證...
✅ 驗證成功！啟動 7 天每日定點抽樣管線...

📅 正在狙擊日期 [2026-06-21] 中午的 LEO 軌道快照...
  -> 成功抓取 1059 筆在軌物體狀態，開始分桶計算...
📅 正在狙擊日期 [2026-06-20] 中午的 LEO 軌道快照...
  -> 成功抓取 33629 筆在軌物體狀態，開始分桶計算...
📅 正在狙擊日期 [2026-06-19] 中午的 LEO 軌道快照...
  -> 成功抓取 80791 筆在軌物體狀態，開始分桶計算...
📅 正在狙擊日期 [2026-06-18] 中午的 LEO 軌道快照...
  -> 成功抓取 87051 筆在軌物體狀態，開始分桶計算...
📅 正在狙擊日期 [2026-06-17] 中午的 LEO 軌道快照...
  -> 成功抓取 66865 筆在軌物體狀態，開始分桶計算...
📅 正在狙擊日期 [2026-06-16] 中午的 LEO 軌道快照...
  -> 成功抓取 89096 筆在軌物體狀態，開始分桶計算...
📅 正在狙擊日期 [2026-06-15] 中午的 LEO 軌道快照...
  -> 成功抓取 83192 筆在軌物體狀態，開始分桶計算...

📊 【大功告成！透過 7天定點動態抽樣 洗出的 LEO 殼層擁擠事實表】
------------------------------------------------------------------------------------------


,date_key,shell_id,satellite_count,debris_count,total_count,density_score
0,20260621,1,27,25,52,2
1,20260621,2,388,175,563,28
2,20260621,3,195,13,208,10
3,20260621,4,171,41,212,10
4,20260620,1,1051,96,1147,57
5,20260620,2,9220,844,10064,100
6,20260620,3,985,2118,3103,100
7,20260620,4,2394,4029,6423,100
8,20260619,1,1254,105,1359,67
9,20260619,2,11058,988,12046,100


------------------------------------------------------------------------------------------
💾 完美的 7 天連續歷史事實表已落檔：【fact_orbital_density_sampled_7days.csv】


In [51]:
import os
import requests
import pandas as pd
import time
from datetime import datetime, timezone, timedelta
from dotenv import load_dotenv

# 1. 身分驗證
load_dotenv()
IDENTITY_EMAIL = os.getenv("SPACETRACK_USER")
IDENTITY_PASSWORD = os.getenv("SPACETRACK_PASS")

session = requests.Session()
login_url = "https://www.space-track.org/ajaxauth/login"
login_payload = {"identity": IDENTITY_EMAIL, "password": IDENTITY_PASSWORD}

print("📡 正在向 Space-Track 進行安全身分驗證...")
login_res = session.post(login_url, data=login_payload, timeout=15)

if login_res.status_code != 200 or "LOGIN_FAILED" in login_res.text:
    print("❌ 登入身分驗證失敗！")
else:
    print("✅ 驗證成功！啟動【歷史冷數據 + 當日即時熱數據】雙軌管線...\n")
    
    fact_rows = []
    now_utc = datetime.now(timezone.utc)
    shell_bins = [200, 400, 600, 800, 2000]
    shell_labels = [1, 2, 3, 4]
    
    # 2. 核心雙軌迴圈：0 代表今天 (熱數據)，1~6 代表過去幾天 (冷歷史)
    for d in range(7):
        target_day = now_utc - timedelta(days=d)
        date_str = target_day.strftime('%Y-%m-%d')
        dk = int(target_day.strftime('%Y%m%d'))
        
        # 🔀 分流邏輯
        if d == 0:
            # 🔥 今天：直接狙擊即時庫 class/gp，確保拿到 100% 完整當下快照
            print(f"🔥 [🚀 熱路徑] 正在直擊即時庫 class/gp，擷取今日 [{date_str}] 最新在軌快照...")
            query_url = "https://www.space-track.org/basicspacedata/query/class/gp/MEAN_MOTION/>11.25/decay_date/null-val/orderby/NORAD_CAT_ID%20asc"
        else:
            # ❄️ 過去幾天：調閱歷史庫 class/gp_history 的 24 小時滑動窗口
            start_window = f"{date_str}%2000:00:00"
            end_window = f"{date_str}%2023:59:59"
            print(f"❄️ [⏳ 冷歷史] 正在調閱歷史庫 class/gp_history，擷取日期 [{date_str}] 的全天封存...")
            query_url = f"https://www.space-track.org/basicspacedata/query/class/gp_history/EPOCH/{start_window}--{end_window}/MEAN_MOTION/>11.25/decay_date/null-val/orderby/NORAD_CAT_ID%20asc"
            
        # 發送請求
        data_res = session.get(query_url, timeout=45)
        
        if data_res.status_code == 200:
            day_json = data_res.json()
            df_day = pd.DataFrame(day_json)
            
            if df_day.empty:
                print(f"⚠️ {date_str} 該時段無資料，跳過。")
                continue
                
            # 數據特徵工程與清洗
            df_day['APOAPSIS'] = pd.to_numeric(df_day['APOAPSIS'], errors='coerce')
            df_day['PERIAPSIS'] = pd.to_numeric(df_day['PERIAPSIS'], errors='coerce')
            df_day['history_altitude_km'] = ((df_day['APOAPSIS'] + df_day['PERIAPSIS']) / 2.0).round(1)
            df_day['date_key'] = dk

            # ─── 🌟 攔截包含 RCS_SIZE 與進階航太特徵的微觀快照 ───
            # 確保這些核心物理特徵欄位為標準文字或浮點數
            df_day['RCS_SIZE'] = df_day['RCS_SIZE'].fillna('UNKNOWN') # 填補缺失值
            df_day['INCLINATION'] = pd.to_numeric(df_day['INCLINATION'], errors='coerce')
            df_day['ECCENTRICITY'] = pd.to_numeric(df_day['ECCENTRICITY'], errors='coerce')
            df_day['COUNTRY_CODE'] = df_day['COUNTRY_CODE'].fillna('UNKNOWN')
            
            # 定義微觀歷史事實表的黃金欄位組合
            micro_cols = [
                'date_key', 
                'EPOCH', 
                'NORAD_CAT_ID', 
                'OBJECT_NAME', 
                'OBJECT_TYPE', 
                'RCS_SIZE',          # ✨ 新增：體積大小
                'COUNTRY_CODE',      # ✨ 新增：所屬國家
                'INCLINATION',       # ✨ 新增：軌道傾角 (區分功能類型)
                'ECCENTRICITY',      # ✨ 新增：離心率 (看軌道形狀變異)
                'history_altitude_km'
            ]

            # 擷取並寫入微觀歷史事實表 CSV (mode='a' 每日增量追加)
            df_micro_snapshot = df_day[micro_cols]
            output_micro_file = "fact_satellite_7days_history.csv"
            
            df_micro_snapshot.to_csv(
                output_micro_file, 
                mode='a', 
                header=not os.path.exists(output_micro_file), 
                index=False
            )
            
            # 本地記憶體超強去重：不論雷達掃幾次，每顆星當天在實體世界只留最新一筆個體
            df_day = df_day.drop_duplicates(subset=['NORAD_CAT_ID'], keep='last')
            df_day['alt'] = df_day['history_altitude_km'] # 對齊高度欄位
            print(f"  -> [微觀] 已將該日 {len(df_micro_snapshot)} 筆全量雷達個體觀測點落檔。")
            print(f"  -> [宏觀] 成功還原該日太空實體共 {len(df_day)} 顆不重複星體，開始分桶...")
            
            # 高度殼層離散分桶
            df_day['shell_id'] = pd.cut(df_day['alt'], bins=shell_bins, labels=shell_labels)
            
            # 統計降維
            for shell in shell_labels:
                group = df_day[df_day['shell_id'] == shell]
                sat_count = int(group[group['OBJECT_TYPE'] == 'PAYLOAD'].shape[0])
                debris_count = int(group[group['OBJECT_TYPE'] != 'PAYLOAD'].shape[0])
                total_count = sat_count + debris_count
                
                # 擁擠度分數歸一化 (以在軌穩定全量約 12000 顆為對比基準)
                density_score = int(min(100, (total_count / 5500.0) * 100))
                
                fact_rows.append({
                    "date_key": dk,
                    "shell_id": shell,
                    "satellite_count": sat_count,
                    "debris_count": debris_count,
                    "total_count": total_count,
                    "density_score": density_score
                })
        else:
            print(f"❌ 抓取 {date_str} 失敗，HTTP 狀態碼: {data_res.status_code}")
            
        # 友善冷卻
        time.sleep(1)
        
    # 3. 完美大會師
    df_final_star_schema = pd.DataFrame(fact_rows).sort_values(by=['date_key', 'shell_id'], ascending=[False, True])
    
    print("\n📊 【終極大功告成！熱數據與冷歷史雙軌交織的 LEO 殼層事實表】")
    print("-" * 90)
    display(df_final_star_schema.head(12)) # 預覽前三天的四大殼層數據
    print("-" * 90)
    
    # 4. 落地保存
    df_final_star_schema.to_csv("fact_orbital_density_final.csv", index=False)
    print("💾 完美的 7 天無斷點即時事實表已落檔：【fact_orbital_density_final.csv】")

📡 正在向 Space-Track 進行安全身分驗證...
✅ 驗證成功！啟動【歷史冷數據 + 當日即時熱數據】雙軌管線...

🔥 [🚀 熱路徑] 正在直擊即時庫 class/gp，擷取今日 [2026-06-21] 最新在軌快照...
  -> [微觀] 已將該日 28887 筆全量雷達個體觀測點落檔。
  -> [宏觀] 成功還原該日太空實體共 28887 顆不重複星體，開始分桶...
❄️ [⏳ 冷歷史] 正在調閱歷史庫 class/gp_history，擷取日期 [2026-06-20] 的全天封存...
  -> [微觀] 已將該日 33656 筆全量雷達個體觀測點落檔。
  -> [宏觀] 成功還原該日太空實體共 20750 顆不重複星體，開始分桶...
❄️ [⏳ 冷歷史] 正在調閱歷史庫 class/gp_history，擷取日期 [2026-06-19] 的全天封存...
  -> [微觀] 已將該日 80791 筆全量雷達個體觀測點落檔。
  -> [宏觀] 成功還原該日太空實體共 24438 顆不重複星體，開始分桶...
❄️ [⏳ 冷歷史] 正在調閱歷史庫 class/gp_history，擷取日期 [2026-06-18] 的全天封存...
  -> [微觀] 已將該日 87051 筆全量雷達個體觀測點落檔。
  -> [宏觀] 成功還原該日太空實體共 23921 顆不重複星體，開始分桶...
❄️ [⏳ 冷歷史] 正在調閱歷史庫 class/gp_history，擷取日期 [2026-06-17] 的全天封存...
  -> [微觀] 已將該日 66865 筆全量雷達個體觀測點落檔。
  -> [宏觀] 成功還原該日太空實體共 20258 顆不重複星體，開始分桶...
❄️ [⏳ 冷歷史] 正在調閱歷史庫 class/gp_history，擷取日期 [2026-06-16] 的全天封存...
  -> [微觀] 已將該日 89096 筆全量雷達個體觀測點落檔。
  -> [宏觀] 成功還原該日太空實體共 23591 顆不重複星體，開始分桶...
❄️ [⏳ 冷歷史] 正在調閱歷史庫 class/gp_history，擷取日期 [2026-06-15] 的全天封存...
  -> [微觀] 已將該日 83192 筆全量雷達個體觀測點落檔。

,date_key,shell_id,satellite_count,debris_count,total_count,density_score
0,20260621,1,1286,180,1466,26
1,20260621,2,11194,1250,12444,100
2,20260621,3,1120,3690,4810,87
3,20260621,4,2530,7586,10116,100
4,20260620,1,1052,96,1148,20
5,20260620,2,9219,844,10063,100
6,20260620,3,985,2119,3104,56
7,20260620,4,2394,4028,6422,100
8,20260619,1,1252,105,1357,24
9,20260619,2,11061,989,12050,100


------------------------------------------------------------------------------------------
💾 完美的 7 天無斷點即時事實表已落檔：【fact_orbital_density_final.csv】


# <span style="background-color: #ffcc00;">**GFZ天氣直灌(需要先跑，dim_time才能有東西)**</span>

In [1]:
import os
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone
from sqlalchemy import create_engine
from dotenv import load_dotenv

print("==================================================")
print("🌤️ LEO 平台核心 - 德國 GFZ Potsdam 每日級別直灌管線")
print(f"當前執行時間 (UTC): {datetime.now(timezone.utc)}")
print("==================================================\n")

# 1. 初始化資料庫連線引擎
load_dotenv()
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "mysecretpassword") 
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "leo_risk_db")

connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

# 2. 自動計算過去 30 天範圍
now_utc = datetime.now(timezone.utc)
start_dt = (now_utc - timedelta(days=30)).strftime('%Y-%m-%dT00:00:00Z')
end_dt = now_utc.strftime('%Y-%m-%dT23:59:59Z')

kp_url = f"https://kp.gfz-potsdam.de/app/json/?start={start_dt}&end={end_dt}&index=Kp"
f107_url = f"https://kp.gfz-potsdam.de/app/json/?start={start_dt}&end={end_dt}&index=Fobs"

try:
    # 3. Extract - 抽取 GFZ 數據
    print("📡 正在向德國 GFZ 伺服器請求 30 天 Kp 指數...")
    kp_res = requests.get(kp_url, timeout=15)
    print("📡 正在向德國 GFZ 伺服器請求 30 天 F10.7 指數...")
    f107_res = requests.get(f107_url, timeout=15)

    if kp_res.status_code == 200 and f107_res.status_code == 200:
        print("✅ GFZ 雙水源 JSON 成功獲取。開始進入 Transform 階段...")
        
        kp_data = kp_res.json()
        f107_data = f107_res.json()

        # Transform - 展平為原始 DataFrame
        df_kp_raw = pd.DataFrame({
            'timestamp_utc': pd.to_datetime(kp_data['datetime']),
            'kp_index': kp_data['Kp']
        })
        
        df_f107_raw = pd.DataFrame({
            'timestamp_utc': pd.to_datetime(f107_data['datetime']),
            'f107_value': f107_data['Fobs']
        })

        # 🌟 核心商業邏輯：收斂顆粒度至「日 (Date)」
        df_kp_raw['date_str'] = df_kp_raw['timestamp_utc'].dt.strftime('%Y-%m-%d')
        df_f107_raw['date_str'] = df_f107_raw['timestamp_utc'].dt.strftime('%Y-%m-%d')

        # Kp 抓每日最大值 (捕捉暴風尖峰)
        df_kp_daily = df_kp_raw.groupby('date_str')['kp_index'].max().reset_index()
        # F10.7 抓每日平均值 (捕捉背景輻射基礎波動)
        df_f107_daily = df_f107_raw.groupby('date_str')['f107_value'].mean().reset_index()
        # 將欄位重新正名，並四捨五入至小數點第一位
        df_f107_daily['f10_7_index'] = df_f107_daily['f107_value'].round(1)

        # 每日級別數據大融合（Join）
        df_daily_weather = pd.merge(df_kp_daily, df_f107_daily[['date_str', 'f10_7_index']], on='date_str', how='inner')

        # 建立時間物件以便後續萃取年月
        df_daily_weather['dt_obj'] = pd.to_datetime(df_daily_weather['date_str'])
        # 生成黃金日期主鍵 date_key (格式: YYYYMMDD)
        df_daily_weather['date_key'] = df_daily_weather['dt_obj'].dt.strftime('%Y%m%d').astype(int)
        
        # 4. 根據每日最大 Kp 評定天氣風險等級
        def get_weather_risk_level(kp):
            if kp >= 5.0: return 'HIGH'
            elif kp >= 3.0: return 'MEDIUM'
            else: return 'LOW'
        df_daily_weather['risk_level'] = df_daily_weather['kp_index'].apply(get_weather_risk_level)

        # ─── 🥇 步驟 A：直灌【日級別時間維度表 dim_time】 ───
        print("\n⏳ 正在清洗並補滿【dim_time 日級別時間維度表】...")
        df_time = pd.DataFrame()
        df_time['date_key'] = df_daily_weather['date_key']
        df_time['year'] = df_daily_weather['dt_obj'].dt.year
        df_time['month'] = df_daily_weather['dt_obj'].dt.month
        df_time['day'] = df_daily_weather['dt_obj'].dt.day
        # 判斷是否為週末 (5 和 6 代表週末)
        df_time['is_weekend'] = df_daily_weather['dt_obj'].dt.weekday.isin([5, 6])
        
        df_time = df_time.drop_duplicates(subset=['date_key'])
        
        try:
            existing_date_keys = pd.read_sql("SELECT date_key FROM dim_time", con=engine)['date_key'].tolist()
            df_time_new = df_time[~df_time['date_key'].isin(existing_date_keys)]
            
            if not df_time_new.empty:
                df_time_new.to_sql('dim_time', con=engine, if_exists='append', index=False, method='multi')
                print(f"  ✅ 成功將 {len(df_time_new)} 天的日期字典補入 dim_time！")
            else:
                print("  ℹ️ dim_time 時間字典已是最新狀態。")
        except Exception as t_err:
            print(f"  ⚠️ dim_time 寫入提示: {t_err}")

        # ─── 🥈 步驟 B：直灌【日級別太空天氣事實表 fact_space_weather】 ───
        print("⏳ 正在將 30 天日級別觀測直灌【fact_space_weather 事实表】...")
        db_weather_cols = ['date_key', 'kp_index', 'f10_7_index', 'risk_level']
        df_weather_final = df_daily_weather[db_weather_cols].copy()
        
        try:
            existing_weathers = pd.read_sql("SELECT date_key FROM fact_space_weather", con=engine)['date_key'].tolist()
            df_weather_new = df_weather_final[~df_weather_final['date_key'].isin(existing_weathers)]
            
            if not df_weather_new.empty:
                df_weather_new.to_sql('fact_space_weather', con=engine, if_exists='append', index=False, method='multi')
                print(f"  🎉 【日級別太空天氣事實表】灌錄大成功！共 {len(df_weather_new)} 筆每日快照安全著陸！")
            else:
                print("  ℹ️ 今日天氣觀測數據先前已存在，自動跳過以防止主鍵衝突。")
        except Exception as w_err:
            print(f"  ❌ [太空天氣事實表] 灌錄失敗: {w_err}")

        print("\n🏁 【德國 GFZ 每日級別天氣直灌管線】全線完畢！請前往 DBeaver 查看！")
        print("==================================================")

    else:
        print(f"❌ GFZ 伺服器請求失敗，狀態碼: Kp={kp_res.status_code}, F107={f107_res.status_code}")
except Exception as e:
    print(f"❌ 執行管線發生連線或解析異常: {e}")

🌤️ LEO 平台核心 - 德國 GFZ Potsdam 每日級別直灌管線
當前執行時間 (UTC): 2026-06-22 12:52:01.687186+00:00

📡 正在向德國 GFZ 伺服器請求 30 天 Kp 指數...
📡 正在向德國 GFZ 伺服器請求 30 天 F10.7 指數...
✅ GFZ 雙水源 JSON 成功獲取。開始進入 Transform 階段...

⏳ 正在清洗並補滿【dim_time 日級別時間維度表】...
  ✅ 成功將 30 天的日期字典補入 dim_time！
⏳ 正在將 30 天日級別觀測直灌【fact_space_weather 事实表】...
  🎉 【日級別太空天氣事實表】灌錄大成功！共 30 筆每日快照安全著陸！

🏁 【德國 GFZ 每日級別天氣直灌管線】全線完畢！請前往 DBeaver 查看！


# <span style="background-color: #ffcc00;">**Dim_satellite、fact_orbital_density、fact_satellite_7days_history 推入 SQL(直接從api整理好推入資料庫)**</span>

In [2]:
import csv
import io
import os
import time
from datetime import datetime, timedelta, timezone

from dotenv import load_dotenv
import pandas as pd
import requests
from sqlalchemy import create_engine, text


# ─── ⚡ PostgreSQL 原生極速二進位 COPY 載入引擎 ───
def psql_insert_copy(table, conn, keys, data_iter):
    raw_conn = (
        conn.connection.dbapi_connection
        if hasattr(conn.connection, "dbapi_connection")
        else conn.connection
    )
    with raw_conn.cursor() as cur:
        s_buf = io.StringIO()
        writer = csv.writer(s_buf)
        writer.writerows(data_iter)
        s_buf.seek(0)
        columns = ", ".join([f'"{k}"' for k in keys])
        cur.copy_expert(
            sql=f"COPY {table.name} ({columns}) FROM STDIN WITH CSV", file=s_buf
        )


# 1. 環境變數與連線初始化
load_dotenv()
IDENTITY_EMAIL = os.getenv("SPACETRACK_USER")
IDENTITY_PASSWORD = os.getenv("SPACETRACK_PASS")

DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "mysecretpassword")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "leo_risk_db")

connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

session = requests.Session()
login_url = "https://www.space-track.org/ajaxauth/login"
login_payload = {"identity": IDENTITY_EMAIL, "password": IDENTITY_PASSWORD}

print("==================================================")
print("📡 LEO 平台核心 - 原汁原味方案 A 潔淨管線 (維度修復版)")
print("==================================================\n")

print("📡 正在向 Space-Track 進行安全身分驗證...")
login_res = session.post(login_url, data=login_payload, timeout=15)

if login_res.status_code != 200 or "LOGIN_FAILED" in login_res.text:
    print("❌ 登入身分驗證失敗！")
else:
    print("✅ 驗證成功！啟動【官方數值 1 Row/Sat/Day 壓縮模型】...\n")

    memory_micro_dfs = []
    memory_dim_sat_dfs = []  # 🌟 新增：專屬的衛星字典維度 Staging 暫存器
    fact_rows = []

    now_utc = datetime.now(timezone.utc)
    shell_bins = [200, 400, 600, 800, 2000]
    shell_labels = [1, 2, 3, 4]

    # 2. 核心雙軌歷史調閱
    for d in range(7):
        target_day = now_utc - timedelta(days=d)
        date_str = target_day.strftime("%Y-%m-%d")
        dk = int(target_day.strftime("%Y%m%d"))

        if d == 0:
            print(f"🔥 [🚀 熱路徑] 正在擷取今日 [{date_str}] 最新在軌快照...")
            query_url = "https://www.space-track.org/basicspacedata/query/class/gp/MEAN_MOTION/>11.25/decay_date/null-val/orderby/NORAD_CAT_ID%20asc"
        else:
            print(f"❄️ [⏳ 冷歷史] 正在調閱歷史 [{date_str}] 的全天觀測點...")
            query_url = f"https://www.space-track.org/basicspacedata/query/class/gp_history/EPOCH/{date_str}%2000:00:00--{date_str}%2023:59:59/MEAN_MOTION/>11.25/decay_date/null-val/orderby/NORAD_CAT_ID%20asc"

        data_res = session.get(query_url, timeout=45)

        if data_res.status_code == 200:
            day_json = data_res.json()
            df_day = pd.DataFrame(day_json)

            if df_day.empty:
                continue

            # 數據強型態與清洗
            df_day["NORAD_CAT_ID"] = df_day["NORAD_CAT_ID"].astype(int)
            df_day["APOAPSIS"] = pd.to_numeric(
                df_day["APOAPSIS"], errors="coerce"
            )
            df_day["PERIAPSIS"] = pd.to_numeric(
                df_day["PERIAPSIS"], errors="coerce"
            )
            df_day["history_altitude_km"] = (
                (df_day["APOAPSIS"] + df_day["PERIAPSIS"]) / 2.0
            ).round(1)
            df_day["date_key"] = int(dk)

            df_day["RCS_SIZE"] = df_day["RCS_SIZE"].fillna("UNKNOWN")
            df_day["INCLINATION"] = pd.to_numeric(
                df_day["INCLINATION"], errors="coerce"
            )
            df_day["ECCENTRICITY"] = pd.to_numeric(
                df_day["ECCENTRICITY"], errors="coerce"
            )
            df_day["COUNTRY_CODE"] = df_day["COUNTRY_CODE"].fillna("UNKNOWN")

            # 方案 A 核心前置：單日、單星去重
            df_day_dedup = df_day.drop_duplicates(
                subset=["NORAD_CAT_ID"], keep="last"
            ).copy()

            # ─── 🌟 關鍵修正：直接在最上游攔截完整的維度字典特徵 ───
            df_sat_dim_keep = df_day_dedup[
                [
                    "NORAD_CAT_ID",
                    "OBJECT_NAME",
                    "OBJECT_TYPE",
                    "COUNTRY_CODE",
                    "RCS_SIZE",
                ]
            ].copy()
            df_sat_dim_keep.columns = [
                "norad_cat_id",
                "object_name",
                "object_type",
                "country_code",
                "rcs_size",
            ]
            memory_dim_sat_dfs.append(df_sat_dim_keep)

            # 收集微觀明細 fakta
            df_micro_keep = df_day_dedup[
                [
                    "date_key",
                    "EPOCH",
                    "NORAD_CAT_ID",
                    "OBJECT_NAME",
                    "OBJECT_TYPE",
                    "history_altitude_km",
                    "INCLINATION",
                    "ECCENTRICITY",
                ]
            ].copy()
            df_micro_keep.columns = [
                "date_key",
                "epoch",
                "norad_cat_id",
                "object_name",
                "object_type",
                "history_altitude_km",
                "inclination",
                "eccentricity",
            ]
            memory_micro_dfs.append(df_micro_keep)

            # 收集宏觀計數
            df_day_dedup["shell_id"] = pd.cut(
                df_day_dedup["history_altitude_km"],
                bins=shell_bins,
                labels=shell_labels,
            )
            for shell in shell_labels:
                grp = df_day_dedup[df_day_dedup["shell_id"] == shell]
                sc = int(grp[grp["OBJECT_TYPE"] == "PAYLOAD"].shape[0])
                dc = int(grp[grp["OBJECT_TYPE"] != "PAYLOAD"].shape[0])
                fact_rows.append(
                    {
                        "date_key": int(dk),
                        "shell_id": int(shell),
                        "satellite_count": sc,
                        "debris_count": dc,
                        "total_count": sc + dc,
                    }
                )
        else:
            print(f"❌ 抓取 {date_str} 失敗")

        time.sleep(1)

    # =========================================================================
    # 3. 🧠 【極速 COPY 落地入庫】
    # =========================================================================
    if memory_micro_dfs and fact_rows and memory_dim_sat_dfs:
        print("\n🧠 [全域工程] 成功在記憶體完成方案 A 矩陣壓縮，啟動 COPY 寫入...")

        df_all_7days_micro = pd.concat(memory_micro_dfs, ignore_index=True)
        df_all_7days_sat_dim = pd.concat(
            memory_dim_sat_dfs, ignore_index=True
        )  # 融合成大字典

        with engine.connect() as conn:
            with conn.begin():

                # ─── 任務 A：動態開闢 dim_time ───
                for d_int in df_all_7days_micro["date_key"].unique():
                    dt_o = datetime.strptime(str(d_int), "%Y%m%d")
                    conn.execute(
                        text(
                            f"INSERT INTO dim_time (date_key, year, month, day, is_weekend) VALUES ({d_int}, {dt_o.year}, {dt_o.month}, {dt_o.day}, {dt_o.weekday() in [5,6]}) ON CONFLICT DO NOTHING"
                        )
                    )

                # ─── 任務 B：🌟 過往歷史物種造冊登錄衛星字典 (修復版！) ───
                # 拿掉舊的強制 UNKNOWN 硬編碼，直接做全域唯一排重
                df_all_7days_sat_dim = df_all_7days_sat_dim.drop_duplicates(
                    subset=["norad_cat_id"], keep="last"
                )

                existing_sats = (
                    pd.read_sql(
                        "SELECT norad_cat_id FROM dim_satellite", con=conn
                    )["norad_cat_id"]
                    .astype(int)
                    .tolist()
                )
                new_sats_to_insert = df_all_7days_sat_dim[
                    ~df_all_7days_sat_dim["norad_cat_id"].isin(existing_sats)
                ]

                if not new_sats_to_insert.empty:
                    print(
                        f"  ⚡ [字典 COPY] 正在同步造冊 {len(new_sats_to_insert)} 顆含有真實國家與尺寸特徵的星體..."
                    )
                    new_sats_to_insert.to_sql(
                        "dim_satellite",
                        con=conn,
                        if_exists="append",
                        index=False,
                        method=psql_insert_copy,
                    )

                # ─── 任務 C：方案 A 絕對主鍵去重與流式 COPY ───
                conn.execute(
                    text("TRUNCATE TABLE fact_satellite_7days_history CASCADE")
                )
                df_all_7days_micro = df_all_7days_micro.drop_duplicates(
                    subset=["date_key", "norad_cat_id"], keep="last"
                )

                print(
                    f"  ⚡ [明細 COPY] 正在流式倒灌方案 A 乾淨明細（共 {len(df_all_7days_micro)} 行）..."
                )
                df_all_7days_micro.to_sql(
                    "fact_satellite_7days_history",
                    con=conn,
                    if_exists="append",
                    index=False,
                    method=psql_insert_copy,
                )
                print("  ✅ [微觀歷史庫] 7 天穩定個體快照全部直灌成功！")

                # ─── 任務 D：精算大盤表 ───
                conn.execute(
                    text("TRUNCATE TABLE fact_orbital_density CASCADE")
                )
                df_macro_final = pd.DataFrame(fact_rows)
                day_sums = df_macro_final.groupby("date_key")[
                    "total_count"
                ].transform("sum")
                df_macro_final["day_leo_total"] = day_sums.replace(
                    0, 1
                ).astype(float)

                df_macro_final["base_occupancy"] = (
                    df_macro_final["total_count"]
                    / df_macro_final["day_leo_total"]
                ) * 100
                df_macro_final["debris_ratio"] = (
                    df_macro_final["debris_count"]
                    / df_macro_final["total_count"]
                ).fillna(0.0)
                df_macro_final["debris_multiplier"] = 1.0 + (
                    df_macro_final["debris_ratio"] * 0.5
                )

                df_macro_final["density_score"] = (
                    df_macro_final["base_occupancy"]
                    * df_macro_final["debris_multiplier"]
                ).round(0).astype(int)
                df_macro_final["density_score"] = df_macro_final[
                    "density_score"
                ].clip(upper=100)

                db_macro_cols = [
                    "date_key",
                    "shell_id",
                    "satellite_count",
                    "debris_count",
                    "total_count",
                    "density_score",
                ]
                df_macro_final[db_macro_cols].to_sql(
                    "fact_orbital_density", con=conn, if_exists="append", index=False
                )
                print("  ✅ [宏觀大盤庫] 每日 LEO 份額加權密度落地成功！")

    print(
        "\n🏁 【特徵維度修復完全勝利】現在 dim_satellite 已經擁有最真實的航太特徵！"
    )

📡 LEO 平台核心 - 原汁原味方案 A 潔淨管線

📡 正在向 Space-Track 進行安全身分驗證...
✅ 驗證成功！啟動【官方數值 1 Row/Sat/Day 壓縮模型】...

🔥 [🚀 熱路徑] 正在擷取今日 [2026-06-22] 最新在軌快照...
❄️ [⏳ 冷歷史] 正在調閱歷史 [2026-06-21] 的全天觀測點...
❄️ [⏳ 冷歷史] 正在調閱歷史 [2026-06-20] 的全天觀測點...
❄️ [⏳ 冷歷史] 正在調閱歷史 [2026-06-19] 的全天觀測點...
❄️ [⏳ 冷歷史] 正在調閱歷史 [2026-06-18] 的全天觀測點...
❄️ [⏳ 冷歷史] 正在調閱歷史 [2026-06-17] 的全天觀測點...
❄️ [⏳ 冷歷史] 正在調閱歷史 [2026-06-16] 的全天觀測點...

🧠 [全域工程] 成功在記憶體完成方案 A 矩陣壓縮，啟動 COPY 寫入...
  ⚡ [字典 COPY] 正在同步造冊 28891 顆星體...
  ⚡ [明細 COPY] 正在流式倒灌方案 A 乾淨明細（共 142946 行）...
  ✅ [微觀歷史庫] 7 天穩定個體快照全部直灌成功！
  ✅ [宏觀大盤庫] 每日 LEO 份額加權密度落地成功！

🏁 【原汁原味方案 A 完全勝利】最純粹的官方數據已完美落袋！請去 DBeaver 點兵！


# <span style="background-color: #ffcc00;">**跨資料源 Join到fact_orbital_risk_assessment**</span>

In [7]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

print("==================================================")
print("🚀 LEO 平台核心 - 終極綜合風險評估合成引擎啟動")
print("==================================================\n")

# 1. 初始化資料庫引擎
load_dotenv()
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "mysecretpassword") 
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "leo_risk_db")

connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

try:
    # 2. Extract - 從資料庫直接讀取最新落地的兩大事實表
    print("📥 正在讀取資料庫中的【宏觀大盤統計事實表】...")
    df_density = pd.read_sql("SELECT date_key, shell_id, total_count, density_score FROM fact_orbital_density", con=engine)
    
    print("📥 正在讀取資料庫中的【日級別太空天氣事實表】...")
    df_weather = pd.read_sql("SELECT date_key, kp_index, f10_7_index FROM fact_space_weather", con=engine)
    
    if df_density.empty or df_weather.empty:
        print("⚠️ 警告：大盤表或天氣表其中之一尚無資料，請先確保步驟一、二已執行完畢！")
    else:
        # 3. Transform - 透過黃金主鍵 date_key 進行跨資料源大融合
        print("🔄 正在進行跨源數據時間軸對齊 (Merge by date_key)...")
        df_risk = pd.merge(df_density, df_weather, on='date_key', how='inner')
        
        # 4. 🔥 核心特徵工程：精算加權綜合風險分數 (Composite Risk Score)
        print("🧠 正在執行太空物理風險特徵工程算分 (加權: 大盤密度 60% + 地磁暴 40%)...")
        # 擁擠度權重 0.6，天氣 Kp 指數轉換為百分制後權重 0.4
        df_risk['composite_risk_score'] = (df_risk['density_score'] * 0.6 + (df_risk['kp_index'] / 9.0 * 100) * 0.4).round(0).astype(int)
        
        # 修正欄位名稱以完美對齊你原本的第七張表結構
        df_risk['orbital_density_score'] = df_risk['density_score']
        df_risk['total_object_count'] = df_risk['total_count']
        df_risk['avg_kp_index'] = df_risk['kp_index']
        df_risk['avg_f10_7_index'] = df_risk['f10_7_index']
        
        # 根據分數打上專業的航太警戒分類標籤
        def get_risk_category(score):
            if score >= 85: return 'CRITICAL'
            elif score >= 60: return 'HIGH'
            elif score >= 30: return 'MEDIUM'
            else: return 'LOW'
        df_risk['risk_category'] = df_risk['composite_risk_score'].apply(get_risk_category)
        
        # 整理要塞入資料庫的欄位
        db_risk_cols = [
            'date_key', 'shell_id', 'orbital_density_score', 'total_object_count', 
            'avg_kp_index', 'avg_f10_7_index', 'composite_risk_score', 'risk_category'
        ]
        df_risk_final = df_risk[db_risk_cols].copy()
        
        # 5. Load - 灌錄【綜合風險評估事實表】
        print("⏳ 正在將融合算分數據直灌【fact_orbital_risk_assessment 表】...")
        try:
            # 讀取現有的主鍵組合防止重複塞入
            existing_risks = pd.read_sql("SELECT date_key, shell_id FROM fact_orbital_risk_assessment", con=engine)
            if not existing_risks.empty:
                # 建立複合主鍵識別證進行排除
                df_risk_final['chk_key'] = df_risk_final['date_key'].astype(str) + "_" + df_risk_final['shell_id'].astype(str)
                existing_risks['chk_key'] = existing_risks['date_key'].astype(str) + "_" + existing_risks['shell_id'].astype(str)
                df_risk_new = df_risk_final[~df_risk_final['chk_key'].isin(existing_risks['chk_key'])].drop(columns=['chk_key'])
            else:
                df_risk_new = df_risk_final
                
            if not df_risk_new.empty:
                df_risk_new.to_sql('fact_orbital_risk_assessment', con=engine, if_exists='append', index=False, method='multi')
                print(f"🎉 🎉 【終極綜合風險評估表】直灌成功！共計 {len(df_risk_new)} 筆殼層風險評估報告安全入庫！")
            else:
                print("ℹ️ 綜合風險評估資料已是最新，無需重覆追加。")
        except Exception as r_err:
            print(f"❌ 綜合風險事實表灌錄失敗: {r_err}")
            
    print("\n🏁 【LEO 核心數據大合體】全線大功告成！你可以用 DBeaver 迎接完美星狀模型了！")
    print("==================================================")
except Exception as e:
    print(f"❌ 執行合體管線發生異常: {e}")

🚀 LEO 平台核心 - 終極綜合風險評估合成引擎啟動

📥 正在讀取資料庫中的【宏觀大盤統計事實表】...
📥 正在讀取資料庫中的【日級別太空天氣事實表】...
🔄 正在進行跨源數據時間軸對齊 (Merge by date_key)...
🧠 正在執行太空物理風險特徵工程算分 (加權: 大盤密度 60% + 地磁暴 40%)...
⏳ 正在將融合算分數據直灌【fact_orbital_risk_assessment 表】...
🎉 🎉 【終極綜合風險評估表】直灌成功！共計 24 筆殼層風險評估報告安全入庫！

🏁 【LEO 核心數據大合體】全線大功告成！你可以用 DBeaver 迎接完美星狀模型了！


# <span style="background-color: #ffcc00;">**daily spacetrack**</span>

In [8]:
import csv
import io
import os
from datetime import datetime, timezone
import requests
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# =============================================================================
# ⚡ 0. 基礎引擎：PostgreSQL 原生極速二進位 COPY 通道
# =============================================================================
def psql_insert_copy(table, conn, keys, data_iter):
    raw_conn = conn.connection.dbapi_connection if hasattr(conn.connection, 'dbapi_connection') else conn.connection
    with raw_conn.cursor() as cur:
        s_buf = io.StringIO()
        writer = csv.writer(s_buf)
        writer.writerows(data_iter)
        s_buf.seek(0)
        columns = ", ".join([f'"{k}"' for k in keys])
        cur.copy_expert(sql=f"COPY {table.name} ({columns}) FROM STDIN WITH CSV", file=s_buf)

# =============================================================================
# 📋 1. 環境初始化與參數鎖定 (2026年日常增量模式)
# =============================================================================
load_dotenv()
IDENTITY_EMAIL = os.getenv("SPACETRACK_USER")
IDENTITY_PASSWORD = os.getenv("SPACETRACK_PASS")

DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "mysecretpassword")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "leo_risk_db")

connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

# 🌟 方案二核心：絕不開迴圈！直接鎖定當下此時此刻的日期門牌
now_utc = datetime.now(timezone.utc)
date_str = now_utc.strftime("%Y-%m-%d")
dk = int(now_utc.strftime("%Y%m%d"))  # 例如：20260622

shell_bins = [200, 400, 600, 800, 2000]
shell_labels = [1, 2, 3, 4]

print("==================================================")
print(f"⏱️ 每日自動化管線啟動！今日目標門牌: [{dk}]")
print("==================================================")

session = requests.Session()
login_url = "https://www.space-track.org/ajaxauth/login"
login_payload = {'identity': IDENTITY_EMAIL, 'password': IDENTITY_PASSWORD}

print("📡 正在向 Space-Track 驗證身分...")
login_res = session.post(login_url, data=login_payload, timeout=15)

if login_res.status_code != 200 or "LOGIN_FAILED" in login_res.text:
    print("❌ 登入身分驗證失敗！排程中止。")
else:
    print("✅ 驗證成功！發動單發「熱路徑快照」狙擊...")
    
    # 🌟 方案二鐵律：只 call 最完整、絕不漏件的 gp 快照端點
    query_url = "https://www.space-track.org/basicspacedata/query/class/gp/MEAN_MOTION/>11.25/decay_date/null-val/orderby/NORAD_CAT_ID%20asc"
    data_res = session.get(query_url, timeout=45)
    
    if data_res.status_code == 200:
        df_day = pd.DataFrame(data_res.json())
        
        if not df_day.empty:
            print(f"📥 成功攔截今日 LEO 活體全量快照：共 {len(df_day)} 筆原始紀錄。")
            
            # ─── 數據清洗與強型態 ───
            df_day["NORAD_CAT_ID"] = df_day["NORAD_CAT_ID"].astype(int)
            df_day["APOAPSIS"] = pd.to_numeric(df_day["APOAPSIS"], errors="coerce")
            df_day["PERIAPSIS"] = pd.to_numeric(df_day["PERIAPSIS"], errors="coerce")
            df_day["history_altitude_km"] = ((df_day["APOAPSIS"] + df_day["PERIAPSIS"]) / 2.0).round(1)
            
            # 🌟 方案二精髓：強行將全量大軍蓋上今天的日期戳記，建立屬於我們自己的歷史！
            df_day["date_key"] = int(dk)
            
            df_day["RCS_SIZE"] = df_day["RCS_SIZE"].fillna("UNKNOWN")
            df_day["INCLINATION"] = pd.to_numeric(df_day["INCLINATION"], errors="coerce")
            df_day["ECCENTRICITY"] = pd.to_numeric(df_day["ECCENTRICITY"], errors="coerce")
            df_day["COUNTRY_CODE"] = df_day["COUNTRY_CODE"].fillna("UNKNOWN")
            
            # 單日單星唯一化
            df_day_dedup = df_day.drop_duplicates(subset=["NORAD_CAT_ID"], keep="last").copy()
            
            # =============================================================================
            # 🧠 2. 資料庫交易發動：實施「冪等性」增量寫入
            # =============================================================================
            with engine.connect() as conn:
                with conn.begin():
                    
                    # ─── 任務 A：動態開闢今天的 dim_time 門牌 ───
                    conn.execute(text(
                        f"INSERT INTO dim_time (date_key, year, month, day, is_weekend) "
                        f"VALUES ({dk}, {now_utc.year}, {now_utc.month}, {now_utc.day}, {now_utc.weekday() in [5,6]}) "
                        f"ON CONFLICT DO NOTHING"
                    ))
                    
                    # ─── 任務 B：今日新登場衛星自動造冊登錄 ───
                    df_sat_dim = df_day_dedup[["NORAD_CAT_ID", "OBJECT_NAME", "OBJECT_TYPE", "COUNTRY_CODE", "RCS_SIZE"]].copy()
                    df_sat_dim.columns = ["norad_cat_id", "object_name", "object_type", "country_code", "rcs_size"]
                    df_sat_dim = df_sat_dim.drop_duplicates(subset=["norad_cat_id"])
                    
                    existing_sats = pd.read_sql("SELECT norad_cat_id FROM dim_satellite", con=conn)["norad_cat_id"].astype(int).tolist()
                    new_sats_to_insert = df_sat_dim[~df_sat_dim["norad_cat_id"].isin(existing_sats)]
                    
                    if not new_sats_to_insert.empty:
                        print(f"  ✨ [字典增量] 發現今日有 {len(new_sats_to_insert)} 顆新發射或首度觀測星體，追加造冊...")
                        new_sats_to_insert.to_sql("dim_satellite", con=conn, if_exists='append', index=False, method=psql_insert_copy)
                    
                    # ─── 任務 C：微觀明細表 ─── 🌟 冪等性防禦：先精準刪除今天，再直灌今天
                    conn.execute(text(f"DELETE FROM fact_satellite_7days_history WHERE date_key = {dk}"))
                    
                    df_micro_keep = df_day_dedup[["date_key", "EPOCH", "NORAD_CAT_ID", "OBJECT_NAME", "OBJECT_TYPE", "history_altitude_km", "INCLINATION", "ECCENTRICITY"]].copy()
                    df_micro_keep.columns = ["date_key", "epoch", "norad_cat_id", "object_name", "object_type", "history_altitude_km", "inclination", "eccentricity"]
                    
                    print(f"  ⚡ [明細 COPY] 正在增量寫入今日微觀快照（共 {len(df_micro_keep)} 行）...")
                    df_micro_keep.to_sql("fact_satellite_7days_history", con=conn, if_exists='append', index=False, method=psql_insert_copy)
                    
                    # ─── 任務 D：宏觀大盤統計 ─── 🌟 冪等性防禦：先精準刪除今天，再附加今天
                    conn.execute(text(f"DELETE FROM fact_orbital_density WHERE date_key = {dk}"))
                    
                    df_day_dedup["shell_id"] = pd.cut(df_day_dedup["history_altitude_km"], bins=shell_bins, labels=shell_labels)
                    fact_rows = []
                    for shell in shell_labels:
                        grp = df_day_dedup[df_day_dedup["shell_id"] == shell]
                        sc = int(grp[grp["OBJECT_TYPE"] == "PAYLOAD"].shape[0])
                        dc = int(grp[grp["OBJECT_TYPE"] != "PAYLOAD"].shape[0])
                        fact_rows.append({
                            "date_key": dk, "shell_id": int(shell), "satellite_count": sc, "debris_count": dc, "total_count": sc + dc
                        })
                    
                    df_macro_final = pd.DataFrame(fact_rows)
                    day_sum = df_macro_final["total_count"].sum()
                    df_macro_final["day_leo_total"] = float(day_sum if day_sum > 0 else 1)
                    df_macro_final["base_occupancy"] = (df_macro_final["total_count"] / df_macro_final["day_leo_total"]) * 100
                    df_macro_final["debris_ratio"] = (df_macro_final["debris_count"] / df_macro_final["total_count"]).fillna(0.0)
                    df_macro_final["debris_multiplier"] = 1.0 + (df_macro_final["debris_ratio"] * 0.5)
                    df_macro_final["density_score"] = (df_macro_final["base_occupancy"] * df_macro_final["debris_multiplier"]).round(0).astype(int).clip(upper=100)
                    
                    db_macro_cols = ["date_key", "shell_id", "satellite_count", "debris_count", "total_count", "density_score"]
                    df_macro_final[db_macro_cols].to_sql("fact_orbital_density", con=conn, if_exists='append', index=False)
                    print(f"  ✅ [大盤增量] 今日 4 大高度殼層份額精算落地成功！")

            # =============================================================================
            # 🌤️ 3. 【連動整合】在此處緊接著 call 你的 GFZ 天氣日常更新與風險評估精算
            # =============================================================================
            print("\n🌦️ 正在啟動下游連動：更新今日太空天氣與綜合風險事實表...")
            # (日後你可以直接把第三步的「今日風險評估程式碼」封裝成 function 貼在這裡，讓它一鍵串聯！)
            
            print(f"\n🏁 【方案二日常更新大獲全勝】今日門牌 {dk} 數據已完美附加！硬碟歷史安全無虞！")
    else:
        print(f"❌ 呼叫 Space-Track 失敗，狀態碼: {data_res.status_code}")

⏱️ 每日自動化管線啟動！今日目標門牌: [20260622]
📡 正在向 Space-Track 驗證身分...
✅ 驗證成功！發動單發「熱路徑快照」狙擊...
📥 成功攔截今日 LEO 活體全量快照：共 28887 筆原始紀錄。
  ⚡ [明細 COPY] 正在增量寫入今日微觀快照（共 28887 行）...
  ✅ [大盤增量] 今日 4 大高度殼層份額精算落地成功！

🌦️ 正在啟動下游連動：更新今日太空天氣與綜合風險事實表...

🏁 【方案二日常更新大獲全勝】今日門牌 20260622 數據已完美附加！硬碟歷史安全無虞！


# <span style="background-color: #ffcc00;">**daily GFZ**</span>

In [11]:
import os
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

print("==================================================")
print("🌤️ LEO 平台核心 - 德國 GFZ 方案二日常增量覆蓋管線 (TypeFixed)")
print(f"當前執行時間 (UTC): {datetime.now(timezone.utc)}")
print("==================================================")

# 1. 初始化資料庫連線引擎
load_dotenv()
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "mysecretpassword") 
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "leo_risk_db")

connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

# 2. 自動計算過去 30 天範圍
now_utc = datetime.now(timezone.utc)
start_dt = (now_utc - timedelta(days=30)).strftime('%Y-%m-%dT00:00:00Z')
end_dt = now_utc.strftime('%Y-%m-%dT23:59:59Z')

kp_url = f"https://kp.gfz-potsdam.de/app/json/?start={start_dt}&end={end_dt}&index=Kp"
f107_url = f"https://kp.gfz-potsdam.de/app/json/?start={start_dt}&end={end_dt}&index=Fobs"

try:
    # 3. Extract - 抽取 GFZ 數據
    print("📡 正在向德國 GFZ 伺服器請求 30 天 Kp 指數...")
    kp_res = requests.get(kp_url, timeout=15)
    print("📡 正在向德國 GFZ 伺服器請求 30 天 F10.7 指數...")
    f107_res = requests.get(f107_url, timeout=15)

    if kp_res.status_code == 200 and f107_res.status_code == 200:
        print("✅ GFZ 雙水源 JSON 成功獲取。開始進入 Transform 階段...")
        
        kp_data = kp_res.json()
        f107_data = f107_res.json()

        # Transform - 展平為原始 DataFrame
        df_kp_raw = pd.DataFrame({
            'timestamp_utc': pd.to_datetime(kp_data['datetime']),
            'kp_index': kp_data['Kp']
        })
        
        df_f107_raw = pd.DataFrame({
            'timestamp_utc': pd.to_datetime(f107_data['datetime']),
            'f107_value': f107_data['Fobs']
        })

        # 核心商業邏輯：收斂顆粒度至「日 (Date)」
        df_kp_raw['date_str'] = df_kp_raw['timestamp_utc'].dt.strftime('%Y-%m-%d')
        df_f107_raw['date_str'] = df_f107_raw['timestamp_utc'].dt.strftime('%Y-%m-%d')

        # Kp 抓每日最大值 (捕捉暴風尖峰)
        df_kp_daily = df_kp_raw.groupby('date_str')['kp_index'].max().reset_index()
        # F10.7 抓每日平均值
        df_f107_daily = df_f107_raw.groupby('date_str')['f107_value'].mean().reset_index()
        df_f107_daily['f10_7_index'] = df_f107_daily['f107_value'].round(1)

        # 每日級別數據大融合（Join）
        df_daily_weather = pd.merge(df_kp_daily, df_f107_daily[['date_str', 'f10_7_index']], on='date_str', how='inner')

        df_daily_weather['dt_obj'] = pd.to_datetime(df_daily_weather['date_str'])
        df_daily_weather['date_key'] = df_daily_weather['dt_obj'].dt.strftime('%Y%m%d').astype(int)
        
        def get_weather_risk_level(kp):
            if kp >= 5.0: return 'HIGH'
            elif kp >= 3.0: return 'MEDIUM'
            else: return 'LOW'
        df_daily_weather['risk_level'] = df_daily_weather['kp_index'].apply(get_weather_risk_level)

        # ─── 資料庫交易 (Transaction) 鎖定發動 ───
        with engine.connect() as conn:
            with conn.begin():

                # ─── 🥇 步驟 A：日常維護【dim_time 時間維度表】 ───
                print("\n⏳ 正在動態維護【dim_time 時間維度表】...")
                df_time = pd.DataFrame()
                df_time['date_key'] = df_daily_weather['date_key']
                df_time['year'] = df_daily_weather['dt_obj'].dt.year
                df_time['month'] = df_daily_weather['dt_obj'].dt.month
                df_time['day'] = df_daily_weather['dt_obj'].dt.day
                df_time['is_weekend'] = df_daily_weather['dt_obj'].dt.weekday.isin([5, 6])
                df_time = df_time.drop_duplicates(subset=['date_key'])
                
                existing_date_keys = pd.read_sql("SELECT date_key FROM dim_time", con=conn)['date_key'].tolist()
                df_time_new = df_time[~df_time['date_key'].isin(existing_date_keys)]
                
                if not df_time_new.empty:
                    df_time_new.to_sql('dim_time', con=conn, if_exists='append', index=False, method='multi')
                    print(f"  ✅ 成功將新登場的 {len(df_time_new)} 天時間門牌補入 dim_time！")
                else:
                    print("  ℹ️ dim_time 時間字典已是最新狀態。")

                # ─── 🥈 步驟 B：🌟 方案二核心【fact_space_weather 事实表覆蓋寫入】 ───
                print("⏳ 正在執行方案二增量覆蓋機制...")
                db_weather_cols = ['date_key', 'kp_index', 'f10_7_index', 'risk_level']
                df_weather_final = df_daily_weather[db_weather_cols].copy()
                
                # 🏥 終極修復點：強制使用 int(x) 將 Numpy 型態剔除，洗回純 Python 乾淨數字！
                target_keys = tuple(int(x) for x in df_weather_final['date_key'].unique())
                
                if len(target_keys) == 1:
                    conn.execute(text(f"DELETE FROM fact_space_weather WHERE date_key = {target_keys[0]}"))
                else:
                    conn.execute(text(f"DELETE FROM fact_space_weather WHERE date_key IN {target_keys}"))
                
                # 數據無害增量安全著陸
                df_weather_final.to_sql('fact_space_weather', con=conn, if_exists='append', index=False, method='multi')
                print(f"  🎉 【太空天氣事實表】自動更新覆蓋大成功！共 {len(df_weather_final)} 天觀測值安全著陸！")

        print("\n🏁 【德國 GFZ 方案二日常增量覆蓋管線】全線通關！")
        print("==================================================")

    else:
        print(f"❌ GFZ 伺服器請求失敗，狀態碼: Kp={kp_res.status_code}, F107={f107_res.status_code}")
except Exception as e:
    print(f"❌ 執行管線發生連線或解析異常: {e}")

🌤️ LEO 平台核心 - 德國 GFZ 方案二日常增量覆蓋管線 (TypeFixed)
當前執行時間 (UTC): 2026-06-22 17:07:42.802005+00:00
📡 正在向德國 GFZ 伺服器請求 30 天 Kp 指數...
📡 正在向德國 GFZ 伺服器請求 30 天 F10.7 指數...
✅ GFZ 雙水源 JSON 成功獲取。開始進入 Transform 階段...

⏳ 正在動態維護【dim_time 時間維度表】...
  ℹ️ dim_time 時間字典已是最新狀態。
⏳ 正在執行方案二增量覆蓋機制...
  🎉 【太空天氣事實表】自動更新覆蓋大成功！共 30 天觀測值安全著陸！

🏁 【德國 GFZ 方案二日常增量覆蓋管線】全線通關！
